# SkyData - Family-Aware Federated Coordination

Comment les replicas d'une donnee se repartissent sans serveur central, et ce que
le Federated Learning apporte reellement.

Ce notebook contient :
1. Le simulateur (v2, corrige suite a la revue interne)
2. Les experiences E1-E4 (dont la couche Federated Learning : none / local / federated / oracle)
3. La demo interactive en bas

Execution > Tout executer. Environ 5-8 minutes pour l'ensemble des experiences.

## Partie 1 - Installation

In [ ]:
!pip install numpy matplotlib --quiet

## Partie 2 - Le moteur (v2)
Harbours, familles, strategies, et la couche Federated Learning : la qualite des
harbours est inconnue, chaque famille l'estime par observations bruitees, le
SkyWorker agrege les estimations (FedAvg pondere) sans centraliser les donnees.

In [ ]:
"""Module principal du simulateur."""
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import numpy as np


@dataclass
class Harbour:
    hid: int
    pos: np.ndarray
    capacity: int
    green: float          # qualite reelle, cachee aux agents
    load: int = 0

    @property
    def free(self) -> int:
        return max(0, self.capacity - self.load)

    @property
    def saturation(self) -> float:
        return self.load / self.capacity if self.capacity else 1.0


@dataclass
class SKD:
    sid: int
    family_id: int
    harbour: int


@dataclass
class Family:
    family_id: int
    members: List[SKD]
    # modele local : estimation de green par harbour + nb d'observations
    green_hat: np.ndarray = None
    obs_count: np.ndarray = None


class SkyWorld:
    def __init__(self, n_harbours=25, n_families=12, family_size=4, seed=0,
                 obs_noise=0.15):
        self.env_rng = np.random.default_rng(seed)          # environnement
        self.rng = np.random.default_rng(seed + 10_000)     # strategies
        self.obs_noise = obs_noise
        self.harbours = []
        for i in range(n_harbours):
            self.harbours.append(Harbour(
                hid=i, pos=self.env_rng.uniform(0, 100, size=2),
                capacity=int(self.env_rng.integers(5, 18)),
                green=float(self.env_rng.uniform(0.0, 1.0))))
        P = np.stack([h.pos for h in self.harbours])
        diff = P[:, None, :] - P[None, :, :]
        self.dist = np.sqrt((diff ** 2).sum(axis=-1))
        self.max_dist = max(1.0, float(self.dist.max()))

        # familles groupees au depart (il faut migrer pour se disperser)
        self.families = []
        sid = 0
        centers = self.env_rng.choice(n_harbours, size=n_families, replace=True)
        for fid in range(n_families):
            members = []
            near = np.argsort(self.dist[int(centers[fid])])[:3]
            for _ in range(family_size):
                h = int(self.env_rng.choice(near))
                self.harbours[h].load += 1
                members.append(SKD(sid=sid, family_id=fid, harbour=h))
                sid += 1
            fam = Family(fid, members)
            fam.green_hat = np.full(n_harbours, 0.5)   # prior neutre
            fam.obs_count = np.zeros(n_harbours)
            self.families.append(fam)

        # compteurs de couts
        self.migration_cost = 0.0     # distance totale parcourue
        self.skw_messages = 0         # contacts SKW <-> agents / familles

    def all_skds(self):
        return [m for fam in self.families for m in fam.members]

    def reachable_harbours(self, current, reach_prob):
        return [h.hid for h in self.harbours
                if h.hid == current or self.rng.random() < reach_prob]

    def observe_green(self, fam: Family, hid: int):
        # observation bruitee de la qualite reelle -> mise a jour incrementale
        obs = float(np.clip(self.harbours[hid].green
                            + self.env_rng.normal(0, self.obs_noise), 0, 1))
        fam.obs_count[hid] += 1
        n = fam.obs_count[hid]
        fam.green_hat[hid] += (obs - fam.green_hat[hid]) / n

    def move(self, skd, target):
        if target == skd.harbour or self.harbours[target].free <= 0:
            return
        self.migration_cost += self.dist[skd.harbour, target] / self.max_dist
        self.harbours[skd.harbour].load -= 1
        self.harbours[target].load += 1
        skd.harbour = target


def family_positions(fam, exclude_sid=None):
    return [m.harbour for m in fam.members
            if exclude_sid is None or m.sid != exclude_sid]


def dispersion(world, positions):
    if len(positions) < 2:
        return 0.0
    ds = [world.dist[positions[i], positions[j]] / world.max_dist
          for i in range(len(positions)) for j in range(i + 1, len(positions))]
    return float(np.mean(ds)) if ds else 0.0


def collision_count(world, positions, threshold=0.20):
    c = 0
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            d = world.dist[positions[i], positions[j]] / world.max_dist
            if positions[i] == positions[j] or d < threshold:
                c += 1
    return c


def candidate_score(world, skd, cand, family_future, family_weight,
                    green_estimate=None, learn_family: Optional[Family] = None):
    """Score d'un harbour candidat.
    - cap recompense la capacite LIBRE (correction de la revue)
    - green utilise l'estimation de la famille si fournie (couche FL),
      sinon la vraie valeur (borne superieure "oracle")
    - si learn_family est fourni, evaluer un candidat produit une observation
    """
    h = world.harbours[cand]
    if h.free <= 0 and cand != skd.harbour:
        return -1e9
    cap = 1.0 - h.saturation
    if learn_family is not None:
        world.observe_green(learn_family, cand)
    green = h.green if green_estimate is None else green_estimate[cand]
    move = world.dist[skd.harbour, cand] / world.max_dist
    if family_future:
        fam_disp = float(np.mean([world.dist[cand, p] / world.max_dist
                                  for p in family_future]))
        too_close = sum(1 for p in family_future
                        if world.dist[cand, p] / world.max_dist < 0.20)
    else:
        fam_disp, too_close = 0.5, 0
    return (0.30 * cap + 0.25 * green + family_weight * fam_disp
            - 0.20 * move - 0.35 * too_close)


FW = 0.70   # poids familial unique pour toutes les strategies family-aware


class Strategy:
    name = "base"

    def step(self, world, reach_prob, d=4):
        raise NotImplementedError

    def _sample_candidates(self, world, skd, reach_prob, d):
        reachable = world.reachable_harbours(skd.harbour, reach_prob)
        if not reachable:
            return [skd.harbour]
        if d == "all":
            cand = list(reachable)
        else:
            cand = list(world.rng.choice(reachable,
                                         size=min(d, len(reachable)),
                                         replace=False))
        if skd.harbour not in cand:
            cand.append(skd.harbour)
        return cand


class RandomStrategy(Strategy):
    name = "Random"

    def step(self, world, reach_prob, d=4):
        skds = world.all_skds()
        world.rng.shuffle(skds)
        for skd in skds:
            feas = [h for h in world.reachable_harbours(skd.harbour, reach_prob)
                    if world.harbours[h].free > 0]
            if feas:
                world.move(skd, int(world.rng.choice(feas)))


class SelfishPoC(Strategy):
    name = "Selfish PoC"

    def step(self, world, reach_prob, d=4):
        skds = world.all_skds()
        world.rng.shuffle(skds)
        for skd in skds:
            cands = self._sample_candidates(world, skd, reach_prob, d)
            best = max(cands, key=lambda c: candidate_score(
                world, skd, c, None, 0.0))
            world.move(skd, int(best))


class NaivePoC(Strategy):
    name = "Naive family-aware PoC"

    def step(self, world, reach_prob, d=4):
        proposals = {}
        snap = {fam.family_id: family_positions(fam) for fam in world.families}
        skds = world.all_skds()
        world.rng.shuffle(skds)
        for skd in skds:
            fam_pos = list(snap[skd.family_id])
            if skd.harbour in fam_pos:
                fam_pos.remove(skd.harbour)
            best = max(self._sample_candidates(world, skd, reach_prob, d),
                       key=lambda c: candidate_score(world, skd, c, fam_pos, FW))
            proposals[skd.sid] = int(best)
        for skd in skds:
            world.move(skd, proposals[skd.sid])


class CoordinatedPoC(Strategy):
    """Coordination par SkyWorker : reservation sequentielle au sein de la
    famille. learning_mode pilote la couche FL :
      None        -> score avec la vraie valeur green (oracle, pas d'apprentissage)
      "none"      -> estimation figee au prior 0.5 (aucun apprentissage)
      "local"     -> chaque famille apprend de ses observations, sans partage
      "federated" -> apprentissage local + agregation inter-familles par SKW
    """
    def __init__(self, learning_mode=None, agg_every=5):
        self.learning_mode = learning_mode
        self.agg_every = agg_every
        self._round = 0
        base = "SKW-coordinated"
        if learning_mode is None:
            self.name = base + " (oracle)"
        else:
            self.name = base + f" ({learning_mode})"

    def step(self, world, reach_prob, d=4):
        self._round += 1
        fams = list(world.families)
        world.rng.shuffle(fams)
        for fam in fams:
            reached = [m for m in fam.members if world.rng.random() < reach_prob]
            if not reached:
                continue
            world.skw_messages += len(reached)     # contacts SKW <-> replicas
            reached.sort(key=lambda m: world.harbours[m.harbour].load,
                         reverse=True)
            reserved = [m.harbour for m in fam.members if m not in reached]
            est, learner = self._estimates(fam)
            for skd in reached:
                ff = reserved + [m.harbour for m in reached
                                 if m.sid != skd.sid and m.harbour != skd.harbour]
                best = max(self._sample_candidates(world, skd, reach_prob, d),
                           key=lambda c: candidate_score(
                               world, skd, c, ff, FW,
                               green_estimate=est, learn_family=learner))
                world.move(skd, int(best))
                reserved.append(skd.harbour)
        if self.learning_mode == "federated" and self._round % self.agg_every == 0:
            self._skw_aggregate(world, reach_prob)

    def _estimates(self, fam):
        if self.learning_mode is None:
            return None, None                       # vraie valeur (oracle)
        if self.learning_mode == "none":
            return np.full(len(fam.green_hat), 0.5), None
        return fam.green_hat, fam                   # local / federated

    def _skw_aggregate(self, world, reach_prob):
        # FedAvg pondere par les compteurs d'observations, sur les familles atteintes
        reached = [f for f in world.families if world.rng.random() < reach_prob]
        if len(reached) < 2:
            return
        world.skw_messages += len(reached)          # contacts SKW <-> familles
        counts = np.stack([f.obs_count for f in reached])
        hats = np.stack([f.green_hat for f in reached])
        total = counts.sum(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            global_hat = np.where(total > 0,
                                  (counts * hats).sum(axis=0) / total, 0.5)
        for f in reached:
            # adoption ponderee : une famille bien informee bouge peu
            w = np.clip(f.obs_count / (f.obs_count + 3.0), 0.0, 0.9)
            f.green_hat = w * f.green_hat + (1 - w) * global_hat
            f.obs_count = np.maximum(f.obs_count, total * 0.25)


def metrics(world):
    fam_disp, fam_coll, est_err = [], [], []
    for fam in world.families:
        pos = family_positions(fam)
        fam_disp.append(dispersion(world, pos))
        fam_coll.append(collision_count(world, pos))
        true_green = np.array([h.green for h in world.harbours])
        est_err.append(float(np.mean(np.abs(fam.green_hat - true_green))))
    # qualite reelle des emplacements occupes (le green VRAI, pas l'estime)
    placement_green = float(np.mean(
        [world.harbours[m.harbour].green for m in world.all_skds()]))
    return {
        "mean_dispersion": float(np.mean(fam_disp)),
        "worst_family_dispersion": float(np.min(fam_disp)),
        "collisions": float(np.mean(fam_coll)),
        "estimation_error": float(np.mean(est_err)),
        "placement_green": placement_green,
        "migration_cost": world.migration_cost,
        "skw_messages": world.skw_messages,
    }


## Partie 3 - Les experiences
E1 strategies (parametres egalises) - E2 Federated Learning - E3 quand la
coordination paie (effet de d) - E4 couts (migration, messages).

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
# (classes definies dans la cellule precedente)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25})

SEEDS = 20
ROUNDS = 80
REACH = 0.65


def run(mk, seed, reach=REACH, rounds=ROUNDS, d=4, track=False):
    world = SkyWorld(seed=seed)
    strat = mk()
    hist = []
    for _ in range(rounds):
        strat.step(world, reach_prob=reach, d=d)
        if track:
            hist.append(metrics(world))
    return (hist if track else metrics(world))


def table(rows, keys=("mean_dispersion", "worst_family_dispersion", "collisions")):
    hdr = {"mean_dispersion": "disp", "worst_family_dispersion": "pire",
           "collisions": "coll", "estimation_error": "err.est",
           "placement_green": "green", "migration_cost": "cout.migr",
           "skw_messages": "msgs"}
    print(f"{'Config':<34}" + "".join(f"{hdr[k]:>10}" for k in keys))
    for name, res in rows:
        vals = "".join(f"{np.mean([r[k] for r in res]):>10.3f}" for k in keys)
        print(f"{name:<34}{vals}")


# ------------------------------------------------------------------ E1
print("=" * 70)
print("E1 - Strategies (cap corrige, fw = 0.70 partout)")
print("=" * 70)
E1 = []
for name, mk in [("Random", RandomStrategy),
                 ("Selfish PoC", SelfishPoC),
                 ("Naive family-aware", NaivePoC),
                 ("SKW-coordinated (oracle)", lambda: CoordinatedPoC(None))]:
    E1.append((name, [run(mk, s) for s in range(SEEDS)]))
table(E1)

# ------------------------------------------------------------------ E2
print()
print("=" * 70)
print("E2 - Federated Learning : la connaissance partagee paie-t-elle ?")
print("    (green inconnu ; none = prior fixe, local = famille seule,")
print("     federated = agregation SKW, oracle = borne superieure)")
print("=" * 70)
modes = [("none", "Sans apprentissage"), ("local", "Local seul"),
         ("federated", "Federe (SKW)"), (None, "Oracle (borne sup.)")]
E2_hist = {}
for mode, label in modes:
    E2_hist[label] = [run(lambda: CoordinatedPoC(mode), s, track=True)
                      for s in range(SEEDS)]
rows = [(label, [h[-1] for h in E2_hist[label]]) for _, label in
        [(m, l) for m, l in modes]]
# erreur d'estimation : significative seulement pour les modes qui apprennent
print(f"{'Config':<34}{'disp':>10}{'green':>10}{'err.est':>10}")
for name, res in rows:
    d = np.mean([r["mean_dispersion"] for r in res])
    g = np.mean([r["placement_green"] for r in res])
    if name in ("Local seul", "Federe (SKW)"):
        e = f"{np.mean([r['estimation_error'] for r in res]):>10.3f}"
    else:
        e = f"{'n/a':>10}"   # prior fixe / valeurs vraies : pas d'estimation apprise
    print(f"{name:<34}{d:>10.3f}{g:>10.3f}{e}")

# figure : erreur d'estimation au fil des tours + qualite de placement
COL = {"Sans apprentissage": "#9aa0a6", "Local seul": "#e8710a",
       "Federe (SKW)": "#1a73e8", "Oracle (borne sup.)": "#137333"}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
for label in E2_hist:
    if label == "Oracle (borne sup.)":
        pass  # l'oracle n'apprend pas ; on le trace quand meme pour reference
    arr = np.array([[h["estimation_error"] for h in run_] for run_ in E2_hist[label]])
    m = arr.mean(0)
    a1.plot(m, label=label, color=COL[label], lw=2)
    a1.fill_between(range(len(m)), m - arr.std(0), m + arr.std(0),
                    color=COL[label], alpha=0.10)
    arr2 = np.array([[h["placement_green"] for h in run_] for run_ in E2_hist[label]])
    m2 = arr2.mean(0)
    a2.plot(m2, label=label, color=COL[label], lw=2)
    a2.fill_between(range(len(m2)), m2 - arr2.std(0), m2 + arr2.std(0),
                    color=COL[label], alpha=0.10)
a1.set_xlabel("Tours"); a1.set_ylabel("Erreur d'estimation (bas = mieux)")
a1.set_title("Decouverte de la qualite des harbours")
a2.set_xlabel("Tours"); a2.set_ylabel("Qualite reelle des emplacements (haut = mieux)")
a2.set_title("Effet sur le placement")
a1.legend(frameon=False, fontsize=8); a2.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig("fig_fl_learning.png", bbox_inches="tight"); plt.close(fig)
print("figure : fig_fl_learning.png")

# ------------------------------------------------------------------ E3
print()
print("=" * 70)
print("E3 - Quand la coordination paie-t-elle ? (vs Naive, fw egalise)")
print("=" * 70)
for d in [2, 4, 8, "all"]:
    n = [run(NaivePoC, s, d=d) for s in range(SEEDS)]
    c = [run(lambda: CoordinatedPoC(None), s, d=d) for s in range(SEEDS)]
    diffs = [ci["mean_dispersion"] - ni["mean_dispersion"] for ci, ni in zip(c, n)]
    print(f"d={str(d):>4} | Naive disp {np.mean([r['mean_dispersion'] for r in n]):.3f} "
          f"coll {np.mean([r['collisions'] for r in n]):.2f}  ||  "
          f"Coord disp {np.mean([r['mean_dispersion'] for r in c]):.3f} "
          f"coll {np.mean([r['collisions'] for r in c]):.2f}  ||  "
          f"delta {np.mean(diffs):+.3f}, Coord gagne {sum(x>0 for x in diffs)}/{SEEDS}")

fig, ax = plt.subplots(figsize=(7.4, 4.2))
ds = [2, 4, 8, 16, 25]
nd, cd = [], []
for d in ds:
    dd = d if d < 25 else "all"
    nd.append(np.mean([run(NaivePoC, s, d=dd)["mean_dispersion"] for s in range(12)]))
    cd.append(np.mean([run(lambda: CoordinatedPoC(None), s, d=dd)["mean_dispersion"]
                       for s in range(12)]))
ax.plot(ds, nd, "-o", label="Naive (simultane)", color="#e8710a", lw=2)
ax.plot(ds, cd, "-o", label="SKW-coordinated", color="#137333", lw=2)
ax.set_xlabel("Nombre de harbours consideres par decision (d)")
ax.set_ylabel("Dispersion finale (haut = mieux)")
ax.set_title("La coordination paie quand la vue s'elargit")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig("fig_ablation_d.png", bbox_inches="tight")
plt.close(fig)
print("figure : fig_ablation_d.png")

# ------------------------------------------------------------------ E4
print()
print("=" * 70)
print("E4 - Le prix de la coordination (couts, reach=0.65, d=4)")
print("=" * 70)
E4 = []
for name, mk in [("Naive family-aware", NaivePoC),
                 ("SKW-coordinated (oracle)", lambda: CoordinatedPoC(None)),
                 ("SKW-coordinated (federated)", lambda: CoordinatedPoC("federated"))]:
    E4.append((name, [run(mk, s) for s in range(SEEDS)]))
table(E4, keys=("mean_dispersion", "migration_cost", "skw_messages"))

print()
print("Termine. Figures : fig_fl_learning.png, fig_ablation_d.png")


## Partie 4 - Demo interactive
Boutons Egoiste / Naive / Coordonnee, curseur d'accessibilite, clic sur un
replica = sa vision partielle du reseau.

In [ ]:
from IPython.display import HTML, display
interface_html = r'''<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SkyData — Coordination familiale (démo interactive)</title>
<style>
  :root{
    --bg:#fbf3f6; --panel:#ffffff; --panel2:#fdf8fa; --line:#efd9e3;
    --ink:#3d2231; --soft:#9c7a8c; --accent:#c2547e; --skw:#e0a83c;
    --f0:#c2547e; --f1:#8e5aa8; --f2:#3e96a8; --f3:#e08a4c; --f4:#6ba86b; --f5:#d77a9e;
    --bad:#c0392b;
  }
  *{box-sizing:border-box;margin:0;padding:0}
  body{background:var(--bg);color:var(--ink);font-family:'DM Sans',-apple-system,Segoe UI,Roboto,sans-serif;
    background-image:radial-gradient(circle at 20% 10%,rgba(194,84,126,.06),transparent 42%),
      radial-gradient(circle at 85% 90%,rgba(142,90,168,.05),transparent 46%);min-height:100vh}
  .wrap{max-width:1180px;margin:0 auto;padding:22px 18px 60px}
  header{display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;margin-bottom:6px}
  h1{font-size:23px;font-weight:700;letter-spacing:-.01em}
  .sub{color:var(--soft);font-size:14px}
  .question{color:var(--accent);font-size:13.5px;margin:8px 0 18px;font-style:italic}
  .layout{display:grid;grid-template-columns:1fr 320px;gap:18px}
  @media(max-width:880px){.layout{grid-template-columns:1fr}}
  .stage{background:linear-gradient(160deg,var(--panel),var(--panel2));border:1px solid var(--line);
    border-radius:18px;padding:14px;position:relative;box-shadow:0 6px 24px rgba(61,34,49,.06)}
  svg{width:100%;height:auto;display:block;border-radius:10px}
  .side{display:flex;flex-direction:column;gap:14px}
  .card{background:linear-gradient(160deg,var(--panel),var(--panel2));border:1px solid var(--line);
    border-radius:16px;padding:16px;box-shadow:0 6px 24px rgba(61,34,49,.06)}
  .card h2{font-size:12px;text-transform:uppercase;letter-spacing:.13em;color:var(--soft);margin-bottom:12px;font-weight:600}
  .controls{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px}
  button{background:#fdf6f9;color:var(--ink);border:1px solid var(--line);border-radius:12px;
    padding:8px 13px;font-size:13.5px;cursor:pointer;transition:.15s;font-family:inherit}
  button:hover{border-color:var(--accent);color:#fff}
  button.on{background:var(--accent);color:#ffffff;border-color:var(--accent);font-weight:600}
  .row{display:flex;align-items:center;gap:10px;margin:10px 0;font-size:13.5px;color:var(--soft)}
  input[type=range]{flex:1;accent-color:var(--accent)}
  .metric{display:flex;justify-content:space-between;align-items:baseline;padding:7px 0;border-bottom:1px solid var(--line)}
  .metric:last-child{border-bottom:0}
  .metric .k{color:var(--soft);font-size:13px}
  .metric .v{font-size:18px;font-weight:700;font-variant-numeric:tabular-nums}
  .legend{display:flex;flex-wrap:wrap;gap:10px;margin-top:10px;font-size:12px;color:var(--soft)}
  .dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px;vertical-align:middle}
  .vision{font-size:13px;line-height:1.7}
  .vision .lab{color:var(--soft)}
  .vision .big{font-size:15px;font-weight:600;color:#fff}
  .pill{display:inline-block;padding:1px 8px;border-radius:20px;font-size:11.5px;margin:2px 3px 0 0}
  .pill.vis{background:rgba(70,211,160,.16);color:var(--accent);border:1px solid rgba(70,211,160,.4)}
  .pill.inv{background:rgba(255,93,93,.12);color:var(--bad);border:1px solid rgba(255,93,93,.35)}
  .hint{color:var(--soft);font-size:12.5px;margin-top:8px;font-style:italic}
  .tag{font-size:11px;color:var(--skw)}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>SkyData · Coordination familiale</h1>
    <span class="sub">démo interactive — les copies d'une donnée cherchent où vivre</span>
  </header>
  <div class="question">« Comment les copies d'une même donnée se répartissent-elles sans serveur central et avec une communication limitée ? »</div>

  <div class="layout">
    <div class="stage">
      <svg id="map" viewBox="0 0 640 460"></svg>
      <div class="legend" id="legend"></div>
      <div class="hint">Clique sur une copie pour voir ce qu'elle perçoit (sa vision partielle). L'étoile jaune est le SkyWorker.</div>
    </div>

    <div class="side">
      <div class="card">
        <h2>Stratégie</h2>
        <div class="controls" id="strats"></div>
        <div class="row"><span>Accessibilité</span>
          <input type="range" id="reach" min="20" max="100" value="65">
          <span id="reachv" style="color:var(--ink);width:38px;text-align:right">65%</span></div>
        <div class="controls">
          <button id="play" class="on">⏸ Pause</button>
          <button id="step">▶ Pas à pas</button>
          <button id="reset">↺ Réinitialiser</button>
        </div>
      </div>

      <div class="card">
        <h2>Mesures en direct</h2>
        <div class="metric"><span class="k">Dispersion moyenne</span><span class="v" id="m_disp">—</span></div>
        <div class="metric"><span class="k">Pire famille</span><span class="v" id="m_worst">—</span></div>
        <div class="metric"><span class="k">Collisions</span><span class="v" id="m_coll">—</span></div>
        <div class="metric"><span class="k">Tour</span><span class="v" id="m_round">0</span></div>
      </div>

      <div class="card">
        <h2>Vision de l'agent <span class="tag" id="vis_tag"></span></h2>
        <div class="vision" id="vision">
          <span class="lab">Clique sur une copie dans la carte pour inspecter sa vue partielle du monde.</span>
        </div>
      </div>
    </div>
  </div>
</div>

<script>
// ----------------------------------------------------------------------------
//  Mini-simulateur (mêmes règles que le simulateur Python, en JS pour la démo)
// ----------------------------------------------------------------------------
const FAM_COLORS = ['--f0','--f1','--f2','--f3','--f4','--f5'].map(v=>getComputedStyle(document.documentElement).getPropertyValue(v).trim());
const W=640,H=460, N_HARB=25, N_FAM=12, FAM_SIZE=4;
let rng = mulberry32(7);
function mulberry32(a){return function(){a|=0;a=a+0x6D2B79F5|0;let t=Math.imul(a^a>>>15,1|a);t=t+Math.imul(t^t>>>7,61|t)^t;return((t^t>>>14)>>>0)/4294967296;}}

let harbours=[], families=[], skds=[], round=0, skwPos=null, strat='coordinated', reachP=0.65, selected=null;

function build(){
  rng=mulberry32(7); harbours=[]; families=[]; skds=[]; round=0; selected=null; skwPos=null;
  for(let i=0;i<N_HARB;i++){
    harbours.push({id:i,x:50+rng()*(W-100),y:50+rng()*(H-100),cap:5+Math.floor(rng()*13),green:rng(),load:0});
  }
  let sid=0;
  for(let f=0;f<N_FAM;f++){
    const center=Math.floor(rng()*N_HARB);
    const near=harbours.map(h=>({id:h.id,d:dist(harbours[center],h)})).sort((a,b)=>a.d-b.d).slice(0,3);
    const mem=[];
    for(let k=0;k<FAM_SIZE;k++){
      const h=near[Math.floor(rng()*near.length)].id;
      harbours[h].load++; const s={sid:sid++,fam:f,harb:h}; mem.push(s); skds.push(s);
    }
    families.push({fid:f,members:mem});
  }
}
function dist(a,b){return Math.hypot(a.x-b.x,a.y-b.y);}
const maxD=Math.hypot(W,H);
function reachable(s){const r=[];for(const h of harbours){if(h.id===s.harb||rng()<reachP)r.push(h.id);}return r;}
function score(s,cand,famPos,fw){
  const h=harbours[cand]; if(h.load>=h.cap&&cand!==s.harb)return -1e9;
  const cap=1-h.load/h.cap, green=h.green, move=dist(harbours[s.harb],h)/maxD;
  let fd=0.5,pen=0;
  if(famPos&&famPos.length){fd=famPos.reduce((a,p)=>a+dist(h,harbours[p])/maxD,0)/famPos.length;
    pen=famPos.filter(p=>dist(h,harbours[p])/maxD<0.20).length;}
  return 0.30*cap+0.25*green+fw*fd-0.20*move-0.35*pen;
}
function sampleCands(s){let r=reachable(s);if(!r.length)return[s.harb];
  r=shuffle(r).slice(0,4); if(!r.includes(s.harb))r.push(s.harb); return r;}
function move(s,t){if(t===s.harb)return;if(harbours[t].load>=harbours[t].cap)return;harbours[s.harb].load--;harbours[t].load++;s.harb=t;}

function stepSim(){
  if(strat==='selfish'){
    shuffle(skds).forEach(s=>{const c=sampleCands(s);move(s,argmax(c,x=>score(s,x,null,0)));});
  } else if(strat==='naive'){
    const snap={};families.forEach(f=>snap[f.fid]=f.members.map(m=>m.harb));
    const prop={};shuffle(skds).forEach(s=>{const c=sampleCands(s);
      const fp=snap[s.fam].filter((p,i)=>!(p===s.harb&&snap[s.fam].indexOf(s.harb)===i));
      prop[s.sid]=argmax(c,x=>score(s,x,fp,0.70));});
    skds.forEach(s=>move(s,prop[s.sid]));
  } else { // coordinated (+ fl identique visuellement)
    shuffle(families).forEach(f=>{
      const reached=f.members.filter(()=>rng()<reachP); if(!reached.length)return;
      reached.sort((a,b)=>harbours[b.harb].load-harbours[a.harb].load);
      const reserved=f.members.filter(m=>!reached.includes(m)).map(m=>m.harb);
      // SkyWorker visite : on le place au barycentre des membres atteints
      skwPos={x:reached.reduce((a,m)=>a+harbours[m.harb].x,0)/reached.length,
              y:reached.reduce((a,m)=>a+harbours[m.harb].y,0)/reached.length};
      reached.forEach(s=>{const c=sampleCands(s);
        const ff=reserved.concat(reached.filter(m=>m.sid!==s.sid&&m.harb!==s.harb).map(m=>m.harb));
        move(s,argmax(c,x=>score(s,x,ff,0.70)));reserved.push(s.harb);});
    });
  }
  round++;
}
function shuffle(a){a=a.slice();for(let i=a.length-1;i>0;i--){const j=Math.floor(rng()*(i+1));[a[i],a[j]]=[a[j],a[i]];}return a;}
function argmax(arr,f){let best=arr[0],bv=-1e9;for(const x of arr){const v=f(x);if(v>bv){bv=v;best=x;}}return best;}

// métriques
function famDisp(f){const p=f.members.map(m=>m.harb);let s=0,n=0;
  for(let i=0;i<p.length;i++)for(let j=i+1;j<p.length;j++){s+=dist(harbours[p[i]],harbours[p[j]])/maxD;n++;}return n?s/n:0;}
function collisions(f){const p=f.members.map(m=>m.harb);let c=0;
  for(let i=0;i<p.length;i++)for(let j=i+1;j<p.length;j++){if(p[i]===p[j]||dist(harbours[p[i]],harbours[p[j]])/maxD<0.2)c++;}return c;}
function metrics(){const d=families.map(famDisp);return{
  disp:avg(d),worst:Math.min(...d),coll:avg(families.map(collisions))};}
const avg=a=>a.reduce((x,y)=>x+y,0)/a.length;

// ----------------------------------------------------------------------------
//  Rendu SVG
// ----------------------------------------------------------------------------
const NS='http://www.w3.org/2000/svg';
function el(t,a){const e=document.createElementNS(NS,t);for(const k in a)e.setAttribute(k,a[k]);return e;}
function cssv(v){return getComputedStyle(document.documentElement).getPropertyValue(v).trim();}

function render(){
  const svg=document.getElementById('map');svg.innerHTML='';
  // liens entre harbours proches (réseau)
  for(let i=0;i<harbours.length;i++)for(let j=i+1;j<harbours.length;j++){
    if(dist(harbours[i],harbours[j])<160){svg.appendChild(el('line',{x1:harbours[i].x,y1:harbours[i].y,x2:harbours[j].x,y2:harbours[j].y,stroke:cssv('--line'),'stroke-width':1,opacity:0.5}));}
  }
  // vision de l'agent sélectionné : surligner visibles/invisibles
  let visibleSet=null;
  if(selected){visibleSet=new Set(reachableStable(selected));}
  // harbours
  harbours.forEach(h=>{
    let op=0.9, stroke=cssv('--line');
    if(visibleSet){ op = visibleSet.has(h.id)?1:0.18; stroke=visibleSet.has(h.id)?cssv('--accent'):cssv('--line');}
    svg.appendChild(el('circle',{cx:h.x,cy:h.y,r:13,fill:'#ffffff',stroke,'stroke-width':visibleSet&&visibleSet.has(h.id)?2:1,opacity:op}));
    // jauge de remplissage
    const fill=Math.min(1,h.load/h.cap);
    svg.appendChild(el('circle',{cx:h.x,cy:h.y,r:13,fill:'none',stroke:cssv('--soft'),'stroke-width':3,
      'stroke-dasharray':`${fill*2*Math.PI*13} ${2*Math.PI*13}`,transform:`rotate(-90 ${h.x} ${h.y})`,opacity:op*0.6}));
  });
  // copies (SKD)
  skds.forEach(s=>{
    const h=harbours[s.harb];
    // léger éclatement si plusieurs copies sur le même harbour
    const same=skds.filter(o=>o.harb===s.harb);const idx=same.indexOf(s);
    const ang=(idx/Math.max(1,same.length))*2*Math.PI;const off=same.length>1?9:0;
    const cx=h.x+Math.cos(ang)*off, cy=h.y+Math.sin(ang)*off;
    let op=1; if(visibleSet&&s!==selected&&!(selected&&s.fam===selected.fam))op=0.85;
    const c=el('circle',{cx,cy,r:selected===s?7:5,fill:FAM_COLORS[s.fam%6],opacity:op,
      stroke:selected===s?'#fff':'none','stroke-width':2,style:'cursor:pointer'});
    c.addEventListener('click',()=>{selected=s;renderVision();render();});
    svg.appendChild(c);
  });
  // SkyWorker
  if(skwPos&&strat==='coordinated'){
    svg.appendChild(el('path',{d:star(skwPos.x,skwPos.y,5,9,4.2),fill:cssv('--skw'),opacity:0.92,stroke:'#7a5b00','stroke-width':0.6}));
  }
  // mesures
  const m=metrics();
  document.getElementById('m_disp').textContent=m.disp.toFixed(3);
  document.getElementById('m_worst').textContent=m.worst.toFixed(3);
  document.getElementById('m_coll').textContent=m.coll.toFixed(2);
  document.getElementById('m_round').textContent=round;
  document.getElementById('m_coll').style.color = m.coll>0.5?cssv('--bad'):cssv('--accent');
}
function star(cx,cy,spikes,outer,inner){let r='',rot=Math.PI/2*3,step=Math.PI/spikes;
  for(let i=0;i<spikes;i++){r+=(i?'L':'M')+(cx+Math.cos(rot)*outer)+','+(cy+Math.sin(rot)*outer)+' ';rot+=step;
    r+='L'+(cx+Math.cos(rot)*inner)+','+(cy+Math.sin(rot)*inner)+' ';rot+=step;}return r+'Z';}

// vision figée pour l'affichage (ne pas reconsommer le rng de la sim)
let _visCache=null;
function reachableStable(s){ if(_visCache&&_visCache.sid===s.sid)return _visCache.vis;
  const vr=mulberry32(s.sid*131+round); const v=[]; for(const h of harbours){if(h.id===s.harb||vr()<reachP)v.push(h.id);} _visCache={sid:s.sid,vis:v};return v;}
function renderVision(){
  const d=document.getElementById('vision');document.getElementById('vis_tag').textContent=selected?('SKD '+selected.sid):'';
  if(!selected){d.innerHTML='<span class="lab">Clique sur une copie dans la carte pour inspecter sa vue partielle du monde.</span>';return;}
  _visCache=null;const vis=reachableStable(selected);
  const fam=families[selected.fam];const famPos=fam.members.filter(m=>m.sid!==selected.sid).map(m=>m.harb);
  const cands=vis.slice(); const decision=argmaxStable(selected,cands,famPos);
  const visPills=vis.map(h=>`<span class="pill vis">H${h}</span>`).join('');
  const invPills=harbours.filter(h=>!vis.includes(h.id)).map(h=>`<span class="pill inv">H${h.id}</span>`).join('');
  d.innerHTML=`
    <div><span class="lab">Famille</span> <span class="big" style="color:${FAM_COLORS[selected.fam%6]}">F${selected.fam}</span>
    &nbsp;·&nbsp;<span class="lab">harbour actuel</span> <span class="big">H${selected.harb}</span></div>
    <div style="margin-top:8px"><span class="lab">Harbours visibles</span><br>${visPills||'<span class="lab">aucun</span>'}</div>
    <div style="margin-top:8px"><span class="lab">Harbours invisibles (hors de portée)</span><br>${invPills}</div>
    <div style="margin-top:8px"><span class="lab">Copies connues de sa famille</span> : ${famPos.map(h=>'H'+h).join(', ')||'aucune'}</div>
    <div style="margin-top:10px"><span class="lab">Décision</span> → <span class="big" style="color:var(--accent)">H${decision}</span></div>`;
}
function argmaxStable(s,cands,fp){let best=s.harb,bv=-1e9;for(const x of cands){const v=score(s,x,fp,0.70);if(v>bv){bv=v;best=x;}}return best;}

// ----------------------------------------------------------------------------
//  Contrôles
// ----------------------------------------------------------------------------
const STRATS=[['selfish','Égoïste'],['naive','Naïve'],['coordinated','Coordonnée (SkyWorker)']];
const sc=document.getElementById('strats');
STRATS.forEach(([k,label])=>{const btn=document.createElement('button');btn.textContent=label;btn.dataset.k=k;
  if(k===strat)btn.classList.add('on');
  btn.onclick=()=>{strat=k;[...sc.children].forEach(b=>b.classList.toggle('on',b.dataset.k===k));build();render();renderVision();};
  sc.appendChild(btn);});

const legend=document.getElementById('legend');
legend.innerHTML=FAM_COLORS.map((c,i)=>`<span><span class="dot" style="background:${c}"></span>Familles ${i} & ${i+6}</span>`).join('')
  +`<span><span class="dot" style="background:var(--skw)"></span>SkyWorker</span>`;

document.getElementById('reach').addEventListener('input',e=>{reachP=e.target.value/100;
  document.getElementById('reachv').textContent=e.target.value+'%';});
let playing=true;
document.getElementById('play').onclick=function(){playing=!playing;this.textContent=playing?'⏸ Pause':'▶ Lecture';this.classList.toggle('on',playing);};
document.getElementById('step').onclick=()=>{skwPos=null;stepSim();render();renderVision();};
document.getElementById('reset').onclick=()=>{build();render();renderVision();};

build();render();
let acc=0;
function loop(t){ if(playing){acc++; if(acc%45===0){skwPos=null;stepSim();render();renderVision();}} requestAnimationFrame(loop);}
requestAnimationFrame(loop);
</script>
</body>
</html>
'''
display(HTML(interface_html))

---
Si la demo s'affiche mal dans Colab, ouvrir directement Interface_demo.html dans un navigateur.